In [5]:
import ee
import geemap
from utils import *

initialize()
config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder
last_year = config.last_year

In [6]:
# function to import all features or images from a folder into one collection

def import_folder_features(folder_path, asset_type='fc'):
    # 1. List all assets in the folder
    # returns a list of dictionaries with 'name', 'type', and 'id'
    asset_list = ee.data.listAssets({'parent': folder_path})['assets']

    feature_ids = [a['name'] for a in asset_list]
                       
    if asset_type == 'fc':
        collections = [ee.FeatureCollection(asset_id) for asset_id in feature_ids]
    else:
        collections = [ee.Image(asset_id) for asset_id in feature_ids]
    
    print(f"Found {len(collections)} {asset_type}s.")
    return collections


Show how the standard deviation increases with biomass and cannot be used to filter data by quality

In [7]:
age = ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_secondary_vegetation_age_v1").select("secondary_vegetation_age_2020").rename("age")

biomass = ee.ImageCollection("projects/sat-io/open-datasets/ESA/ESA_CCI_AGB").filterDate('2020-01-01','2021-01-01').mean().select("AGB").rename("biomass")
sd = ee.ImageCollection("projects/sat-io/open-datasets/ESA/ESA_CCI_AGB").filterDate('2020-01-01','2021-01-01').first().select("SD").rename("sd")

amazon = ee.FeatureCollection("projects/extents-490617/assets/biomes_br").filter(ee.Filter.eq('Bioma', 'Amazônia'))

## Keep only patches of the same age and greater than 1ha

Do not exclude edge pixels. Selecting exclusively based on area and age.

In [8]:
grid = amazon.geometry().coveringGrid('EPSG:4326', 100000)

grid_list = grid.toList(grid.size())
n = grid.size().getInfo()

for i in range(n):
    cell = ee.Feature(grid_list.get(i))

    vectors = age.reduceToVectors(
        geometry=cell.geometry(),
        geometryType='polygon',
        scale=30,
        eightConnected=True,
        maxPixels=1e12,
        labelProperty='age',
        tileScale = 16
    )

    vectors = vectors.map(
        lambda f: f.set({
            'area_m2': f.geometry().area(maxError=1),
            'tile_id': i + 1
        })
    ).filter(ee.Filter.gt('area_m2', 10000)).select(['age', 'tile_id'])

    task = ee.batch.Export.table.toAsset(
        collection=vectors,
        description=f"secondary_age_vectors_{i+1:03d}",
        assetId=f"{data_folder}/secondary_polygons/secondary_age_vectors_{i+1:03d}"
    )
    # task.start()

#309 needs to be run again


# Export the data with GEDI asymptote, GEDI biomass and with the age pixels only with contiguous patches of >1ha of area

In [ ]:
fire = (ee.Image("projects/mapbiomas-public/assets/brazil/fire/collection3/mapbiomas_fire_collection3_annual_burned_coverage_v1")
    .select([f"burned_coverage_{year}" for year in config.range_1985_2020])
    .byte()
    .rename([str(year) for year in config.range_1985_2020])
    .gt(0)
    .reduce('sum').rename("num_fires")).unmask(0)

floodable_forests = (ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1")
        .select(f"classification_{last_year}").eq(6)).rename("floodable_forests")

### ----------------- Surrounding Landscape -----------------

quarters_ecoreg_biomass = ee.Image("projects/forestregrowth/assets/quarters_ecoreg_biomass")
ecoreg = ee.Image("projects/forestregrowth/assets/ecoreg")
distance_deep_forest = ee.Image(f"{data_folder}/distance_deep_forest").rename("dist")
sur_cover = ee.Image(f"{data_folder}/sur_cover")
near_mat_gedi = ee.Image("projects/forestregrowth/assets/nearest_mature_GEDI")

### ----------------- Environmental -----------------

categorical = ee.Image(f"{data_folder}/categorical")
topography = ee.Image("CSP/ERGo/1_0/Global/ALOS_landforms").rename("topography") # 90m resolution
soil = ee.Image(f"{data_folder}/soilgrids")
terraclim = ee.Image(f"{data_folder}/terraclim_1958_2019")

In [25]:
def quality_mask(image):
    image = image.updateMask(image.select('l4_quality_flag').eq(1)) \
              .updateMask(image.select('degrade_flag').eq(0))
    relative_se = image.select('agbd_se').divide(image.select('agbd'))
    return image.updateMask(relative_se.lte(0.5))

GEDI = (ee.ImageCollection('LARSE/GEDI/GEDI04_A_002_MONTHLY')
        .filterDate('2020-01-01', '2020-12-31')
             .map(quality_mask)
             .select(['agbd']))

GEDI = GEDI.mean().toInt16().rename('GEDI_biomass')

all_features = import_folder_features(f"{data_folder}/secondary_polygons")
feature_collection = ee.FeatureCollection(all_features).flatten()

unified_secondary = ee.Image.cat([age, fire, floodable_forests, quarters_ecoreg_biomass, ecoreg, distance_deep_forest, sur_cover, categorical, topography, soil, terraclim, GEDI]).updateMask(GEDI)


Found 482 fcs.


In [ ]:
def export_csv(image, n_chunks = 1):

    properties_to_export = image.bandNames().getInfo()
    
    total_features = feature_collection.size().getInfo()
    chunk_size = int(total_features * 1/n_chunks)

    def process_chunk(chunk_index):
        start = chunk_index * chunk_size
        chunk = feature_collection.toList(chunk_size, start)
        selected_pixels = ee.FeatureCollection(chunk)

        unified_fc = image.reduceRegions(selected_pixels, ee.Reducer.first(), 30)

        task = ee.batch.Export.table.toDrive(
            collection = unified_fc,
            description = f"gedi_sampled_{chunk_index}",
            fileFormat = "CSV",
            selectors = [p for p in properties_to_export if p not in ['system:index', '.geo']]
        )
        task.start()

    for i in range(n_chunks):
        if i*chunk_size < total_features:
            process_chunk(i)

export_csv(unified_secondary, 20)

In [ ]:
all_images = import_folder_features(f"{data_folder}/GEDI_mature", asset_type = 'image')
mature_image = ee.ImageCollection(all_images).mosaic()

feature_collection = ee.FeatureCollection(f"{data_folder}/grid_10k_amazon_secondary")

def buffer_feature(feature):
    distance = feature.getNumber('first').add(10000)
    buffer = feature.geometry().buffer(distance)
    return feature.setGeometry(buffer)

edge_detec = mature_image.unmask(-1).zeroCrossing()
distance_to_10k_forest = edge_detec.fastDistanceTransform(100, 'pixels').sqrt() \
    .multiply(ee.Image.pixelArea().sqrt()).add(10000).toInt32().rename("distance_to_10k_forest")

processed_fc = distance_to_10k_forest.reduceRegions(
    collection = feature_collection,
    reducer = ee.Reducer.first(),
    scale = 10000
)

# Buffer each point to reach the nearest pixel
buffered_features = processed_fc.map(buffer_feature)

# Extract the biomass value for each buffered region
# This will get the value from nearest valid pixel
nearest_mature = mature_image.reduceRegions(
    collection = buffered_features,
    reducer = ee.Reducer.firstNonNull(),
    scale = 10000,
    tileScale = 16
).map(lambda feature: feature.centroid())

nearest_mature_image = nearest_mature.reduceToImage(
    properties=['first'],
    reducer=ee.Reducer.first()
).unmask(0)

# nearest_mature_image = nearest_mature_image.add(mature_image.unmask(0)).selfMask().rename("nearest_mature")

# # smooth it out to avoid sharp changes in expected biomass for remote areas far from mature forests
# uniform_kernel = ee.Kernel.square(radius = 3, units = 'pixels')
# nearest_mature_image = nearest_mature_image.reduceNeighborhood(reducer = ee.Reducer.mean(), kernel = uniform_kernel).rename("nearest_mature")

# # export_image(nearest_mature_image, f"nearest_mature{suffix}", region = roi, scale = 10000)

# ee.batch.Export.image.toAsset(
#     image = nearest_mature_image,
#     assetId = "projects/forestregrowth/assets/nearest_mature_image_GEDI",
#     scale = 10000
# ).start()

Found 148 images.
